In [1]:
%pip -q install Flask Flask-Login Flask-WTF

In [2]:
import os
import sqlite3
import threading
import time
import secrets

from flask import (
    Flask, request, redirect, url_for,
    render_template_string, flash
)
from flask_login import (
    LoginManager, UserMixin, login_user,
    login_required, logout_user, current_user
)
from flask_wtf.csrf import CSRFProtect
from werkzeug.security import generate_password_hash, check_password_hash
from IPython.display import display, HTML
from google.colab import output

# =========================================================
# 1. KONFIGURASI APLIKASI
# =========================================================

app = Flask(__name__)
app.config["SECRET_KEY"] = secrets.token_hex(32)
app.config["SESSION_COOKIE_HTTPONLY"] = True
app.config["SESSION_COOKIE_SAMESITE"] = "Lax"

csrf = CSRFProtect(app)

login_manager = LoginManager()
login_manager.init_app(app)
login_manager.login_view = "login"

DB_PATH = "/content/URWEB_admin.db"


# =========================================================
# 2. DATABASE SQLITE (PENJUALAN, PELANGGAN, ADMIN, SETTINGS)
# =========================================================

def get_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    with get_db() as conn:
        # Tabel Admins
        conn.execute("""
            CREATE TABLE IF NOT EXISTS admins (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL
            )
        """)

        # Tabel Data Penjualan
        conn.execute("""
            CREATE TABLE IF NOT EXISTS penjualan (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                kode_trx TEXT NOT NULL,
                nama_pelanggan TEXT NOT NULL,
                produk TEXT NOT NULL,
                jumlah INTEGER NOT NULL,
                total_harga REAL NOT NULL,
                tanggal TEXT NOT NULL,
                status TEXT NOT NULL
            )
        """)

        # Tabel Data Pelanggan
        conn.execute("""
            CREATE TABLE IF NOT EXISTS pelanggan (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                kode_pelanggan TEXT UNIQUE NOT NULL,
                nama TEXT NOT NULL,
                email TEXT NOT NULL,
                telepon TEXT NOT NULL,
                alamat TEXT NOT NULL,
                tanggal_daftar TEXT NOT NULL
            )
        """)

        # Tabel Pengaturan Sistem
        conn.execute("""
            CREATE TABLE IF NOT EXISTS settings (
                id INTEGER PRIMARY KEY CHECK (id = 1),
                nama_aplikasi TEXT NOT NULL,
                versi TEXT NOT NULL,
                footer_text TEXT NOT NULL
            )
        """)

        # Akun Admin Default
        existing = conn.execute(
            "SELECT id FROM admins WHERE username = ?",
            ("admin",)
        ).fetchone()

        if existing is None:
            conn.execute(
                "INSERT INTO admins (username, password_hash) VALUES (?, ?)",
                ("admin", generate_password_hash("GantiPassword123!"))
            )

        # Pengaturan Default
        existing_set = conn.execute("SELECT id FROM settings WHERE id = 1").fetchone()
        if existing_set is None:
            conn.execute("""
                INSERT INTO settings (id, nama_aplikasi, versi, footer_text)
                VALUES (1, 'UR WEB Admin Portal', 'v2.5 Pro', 'UR WEB Admin Portal © 2026')
            """)

        # Data Penjualan Contoh
        existing_sales = conn.execute("SELECT id FROM penjualan").fetchone()
        if existing_sales is None:
            sales_samples = [
                ('#TRX-001', 'Budi Santoso', 'Laptop Asus Zenbook', 1, 14500000, '2026-03-25', 'Selesai'),
                ('#TRX-002', 'Siti Rahma', 'Mouse Wireless Logistics', 2, 450000, '2026-03-26', 'Selesai'),
                ('#TRX-003', 'Andi Wijaya', 'Keyboard Mechanical RGB', 1, 850000, '2026-03-27', 'Pending'),
            ]
            conn.executemany("""
                INSERT INTO penjualan (kode_trx, nama_pelanggan, produk, jumlah, total_harga, tanggal, status)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, sales_samples)

        # Data Pelanggan Contoh
        existing_cust = conn.execute("SELECT id FROM pelanggan").fetchone()
        if existing_cust is None:
            cust_samples = [
                ('CUST-001', 'Budi Santoso', 'budi@gmail.com', '081234567890', 'Tangerang', '2026-01-10'),
                ('CUST-002', 'Siti Rahma', 'siti@yahoo.com', '089876543210', 'Jakarta', '2026-02-14'),
                ('CUST-003', 'Andi Wijaya', 'andi@hotmail.com', '085611223344', 'Bandung', '2026-03-01'),
            ]
            conn.executemany("""
                INSERT INTO pelanggan (kode_pelanggan, nama, email, telepon, alamat, tanggal_daftar)
                VALUES (?, ?, ?, ?, ?, ?)
            """, cust_samples)


init_db()


# =========================================================
# 3. MODEL ADMIN FLASK-LOGIN
# =========================================================

class Admin(UserMixin):
    def __init__(self, row):
        self.id = str(row["id"])
        self.username = row["username"]
        self.password_hash = row["password_hash"]

    @staticmethod
    def find_by_id(admin_id):
        with get_db() as conn:
            row = conn.execute(
                "SELECT * FROM admins WHERE id = ?",
                (admin_id,)
            ).fetchone()
        return Admin(row) if row else None

    @staticmethod
    def find_by_username(username):
        with get_db() as conn:
            row = conn.execute(
                "SELECT * FROM admins WHERE username = ?",
                (username,)
            ).fetchone()
        return Admin(row) if row else None


@login_manager.user_loader
def load_user(user_id):
    return Admin.find_by_id(user_id)


# =========================================================
# 4. TEMPLATE HTML LOGIN
# =========================================================

LOGIN_HTML = r"""
<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Universitas Raharja WEB | Login Penjualan</title>

<style>
* { box-sizing: border-box; }
:root { --bg: #000000; --panel: #000000; --blue: #000000; --cyan: #000000; --text: #000000; --muted: #000000; }

body {
    margin: 0; min-height: 100vh; display: grid; place-items: center; padding: 28px;
    font-family: Inter, Arial, sans-serif; color: var(--text);
    background: radial-gradient(circle at 15% 15%, #000000 0, transparent 35%),
                radial-gradient(circle at 90% 90%, #142a4b 0, transparent 30%), var(--bg);
}

.container {
    width: min(100%, 1020px); display: grid; grid-template-columns: 1fr 1fr;
    overflow: hidden; border: 1px solid #293b59; border-radius: 24px;
    background: linear-gradient(145deg, #ADD8E6, #FFFFFF); box-shadow: 0 28px 80px #0006;
}

.hero { padding: 48px; background: linear-gradient(145deg, #ADD8E6, #C8A2C8); position: relative; }
.logo { color: var(--cyan); font-size: 32px; font-weight: 900; letter-spacing: 2px; }
.tag { margin-top: 50px; color: #000000; font-size: 11px; font-weight: 800; letter-spacing: 2px; }
.hero h1 { font-size: clamp(30px, 4vw, 42px); line-height: 1.1; margin: 16px 0; }
.hero p { color: var(--muted); line-height: 1.8; max-width: 360px; }

.form-panel { padding: 48px; }
.eyebrow { color: var(--cyan); letter-spacing: 2px; font-size: 11px; font-weight: 800; }
h2 { margin: 14px 0 8px; font-size: 29px; }
.subtitle { color: var(--muted); font-size: 14px; margin-bottom: 28px; }

label { display: block; margin: 18px 0 8px; font-size: 13px; color: #000000; font-weight: 600; }
input { display: block; width: 100%; padding: 14px; color: blue; background: #0a1425; border: 1px solid #304361; border-radius: 11px; outline: none; font-size: 14px; }
input:focus { border-color: var(--blue); box-shadow: 0 0 0 3px #3978f629; }

button {
    width: 100%; margin-top: 25px; padding: 15px; color: white;
    background: linear-gradient(90deg, #FFC5D3, #4c9dff); border: 0;
    border-radius: 11px; font-weight: 800; cursor: pointer; transition: transform .2s;
}
button:hover { transform: translateY(-2px); }
.flash { color:#000000; background: #ADD8E6; padding: 11px; border-radius: 99px; font-size: 13px; margin-bottom: 10px; text-align: center; }
.footer { margin-top: 24px; color: var(--muted); font-size: 13px; }

@media (max-width: 720px) { .container { grid-template-columns: 1fr; } .hero, .form-panel { padding: 30px; } }
</style>
</head>

<body>
<div class="container">
    <section class="hero">
        <div class="logo">UNIVERSITAS RAHARJA</div>
        <div class="tag">SYSTEM PENJUALAN</div>
        <h1>Kelola Data<br>Penjualan Anda.</h1>
        <p>Portal administrator untuk memantau transaksi, pelanggan, dan sistem pengaturan.</p>
    </section>

    <section class="form-panel">
        <div class="eyebrow">WELCOME BACK</div>
        <h2>Login Admin</h2>
        <div class="subtitle">Masukkan kredensial Anda untuk membuka Data Penjualan.</div>

        {% for message in get_flashed_messages() %}
            <div class="flash">{{ message }}</div>
        {% endfor %}

        <form method="POST">
            <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
            <label for="username">Username</label>
            <input id="username" name="username" type="text" placeholder="Masukkan username" required>

            <label for="password">Password</label>
            <input id="password" name="password" type="password" placeholder="Masukkan password" required>

            <button type="submit">Buka Web Penjualan &rarr;</button>
        </form>

        <div class="footer">UR WEB Admin Portal &copy; 2026</div>
    </section>
</div>
</body>
</html>
"""


# =========================================================
# 5. TEMPLATE BASE DASHBOARD MULTI-FITUR
# =========================================================

BASE_LAYOUT = r"""
<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>UR WEB | {{ title }}</title>

<style>
* { box-sizing: border-box; }
body { margin: 0; background: #FFFFFF; color: #000000; font-family: Inter, Arial, sans-serif; }
.layout { display: flex; min-height: 100vh; }

.sidebar {
    width: 245px; flex-shrink: 0; padding: 28px 18px; background: #4682B4; border-right: 1px solid #263751;
}
.logo { color:#FFFFFF; font-size: 26px; font-weight: 900; letter-spacing: 2px; margin: 8px 12px 42px; }
.nav {
    display: block; padding: 14px; margin: 7px 0; border-radius: 10px; color: #FFFFFF;
    font-size: 14px; text-decoration: none; cursor: pointer; transition: background 0.2s;
}
.nav:hover { background: rgba(255, 255, 255, 0.2); }
.nav.active { color: black; background: #FFFFFF; font-weight: bold; }

main { width: 100%; max-width: 1500px; padding: 36px; margin: 0 auto; }
.topbar { display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 20px; }
.kicker { font-size: 11px; letter-spacing: 2px; font-weight: 800; }
h1 { font-size: 32px; margin: 12px 0; }

.actions { display: flex; align-items: center; gap: 14px; }
.badge { color: white; background: #3978f6; border-radius: 99px; padding: 10px 13px; font-size: 12px; }
.logout { color: white; background: #e53e3e; padding: 11px 16px; border-radius: 10px; text-decoration: none; font-size: 13px; font-weight: 700; }

.cards { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 18px; margin: 32px 0; }
.card, .panel { background: #FFC0CB; border: 1px solid #263751; border-radius: 17px; padding: 23px; }
.card-label { font-size: 13px; font-weight: bold; }
.value { font-size: 28px; font-weight: 800; margin: 15px 0 10px; }
.trend { font-size: 12px; }

table { width: 100%; border-collapse: collapse; margin-top: 18px; }
th, td { text-align: left; padding: 14px 10px; border-bottom: 1px solid #263751; font-size: 13px; }
th { color: #000000; font-size: 11px; letter-spacing: 1px; }

.form-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-top: 15px; }
.form-group { display: flex; flex-direction: column; }
.form-group label { font-size: 12px; font-weight: bold; margin-bottom: 5px; }
.form-group input, .form-group select, .form-group textarea { padding: 10px; border-radius: 8px; border: 1px solid #ccc; font-size: 13px; }
.btn-submit { margin-top: 15px; padding: 12px 20px; background: #3978f6; color: white; border: none; border-radius: 8px; font-weight: bold; cursor: pointer; }

.progress-bar { background: #e0e0e0; border-radius: 10px; height: 18px; width: 100%; overflow: hidden; margin-top: 8px; }
.progress-fill { background: #3978f6; height: 100%; text-align: right; padding-right: 8px; color: white; font-size: 11px; line-height: 18px; font-weight: bold; }

.flash-msg { padding: 12px; background: #c6f6d5; color: #22543d; border-radius: 8px; margin-bottom: 20px; font-size: 14px; font-weight: bold; }

@media (max-width: 850px) { .sidebar { display: none; } main { padding: 22px; } .cards { grid-template-columns: 1fr; } .form-grid { grid-template-columns: 1fr; } }
</style>
</head>

<body>
<div class="layout">
    <aside class="sidebar">
        <div class="logo">UR WEB</div>
        <a href="{{ url_for('dashboard') }}" class="nav {% if active_page == 'overview' %}active{% endif %}">&#9638; &nbsp; Overview Dashboard</a>
        <a href="{{ url_for('penjualan') }}" class="nav {% if active_page == 'penjualan' %}active{% endif %}">&#9672; &nbsp; Data Penjualan</a>
        <a href="{{ url_for('pelanggan') }}" class="nav {% if active_page == 'pelanggan' %}active{% endif %}">&#9818; &nbsp; Data Pelanggan</a>
        <a href="{{ url_for('admin_users') }}" class="nav {% if active_page == 'admin_users' %}active{% endif %}">&#9813; &nbsp; Admin Users</a>
        <a href="{{ url_for('settings') }}" class="nav {% if active_page == 'settings' %}active{% endif %}">&#9881; &nbsp; Sistem Pengaturan</a>
    </aside>

    <main>
        <div class="topbar">
            <div>
                <div class="kicker">ADMIN WORKSPACE / {{ kicker }}</div>
                <h1>{{ title }}</h1>
                <div class="muted">{{ description }}</div>
            </div>
            <div class="actions">
                <span class="badge">&#9679; {{ app_settings.nama_aplikasi }}</span>
                <a class="logout" href="{{ url_for('logout') }}">Logout</a>
            </div>
        </div>

        {% for message in get_flashed_messages() %}
            <div class="flash-msg" style="margin-top:20px;">{{ message }}</div>
        {% endfor %}

        <!-- PERUBAHAN FITUR 1: OVERVIEW DASHBOARD UTAMA -->
        {% if active_page == 'overview' %}
        <section class="cards">
            <div class="card">
                <div class="card-label">Total Omset Penjualan</div>
                <div class="value">Rp {{ "{:,.0f}".format(total_omset) }}</div>
                <div class="trend">Akumulasi Seluruh Transaksi</div>
            </div>
            <div class="card">
                <div class="card-label">Total Transaksi Selesai</div>
                <div class="value">{{ total_trx }}</div>
                <div class="trend">Pesanan Berhasil</div>
            </div>
            <div class="card">
                <div class="card-label">Total Pelanggan Aktif</div>
                <div class="value">{{ total_customers }}</div>
                <div class="trend">Pelanggan Terdaftar</div>
            </div>
        </section>

        <section class="panel" style="margin-bottom: 25px;">
            <h2>Analitik Ringkasan Penjualan</h2>
            <p style="font-size: 13px; color: #333;">Performa performa pesanan dan target kuota penjualan sistem.</p>

            <div style="margin-top: 15px;">
                <label style="font-size: 12px; font-weight: bold;">Tingkat Keberhasilan Pesanan (Status Selesai):</label>
                <div class="progress-bar">
                    <div class="progress-fill" style="width: 85%;">85%</div>
                </div>
            </div>

            <div style="margin-top: 15px;">
                <label style="font-size: 12px; font-weight: bold;">Target Capaian Omset Bulanan:</label>
                <div class="progress-bar">
                    <div class="progress-fill" style="width: 70%; background: #48bb78;">70%</div>
                </div>
            </div>
        </section>

        <section class="panel">
            <h2>Transaksi Penjualan Terbaru</h2>
            <table>
                <thead>
                    <tr>
                        <th>KODE TRX</th>
                        <th>NAMA PELANGGAN</th>
                        <th>PRODUK</th>
                        <th>TOTAL HARGA</th>
                        <th>STATUS</th>
                    </tr>
                </thead>
                <tbody>
                    {% for row in recent_sales %}
                    <tr>
                        <td><b>{{ row.kode_trx }}</b></td>
                        <td>{{ row.nama_pelanggan }}</td>
                        <td>{{ row.produk }}</td>
                        <td>Rp {{ "{:,.0f}".format(row.total_harga) }}</td>
                        <td><b>{{ row.status }}</b></td>
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
        </section>

        <!-- FITUR 2: DATA PENJUALAN -->
        {% elif active_page == 'penjualan' %}
        <section class="cards">
            <div class="card">
                <div class="card-label">Total Penjualan</div>
                <div class="value">Rp {{ "{:,.0f}".format(total_omset) }}</div>
                <div class="trend">Akumulasi Omset</div>
            </div>
            <div class="card">
                <div class="card-label">Total Transaksi</div>
                <div class="value">{{ total_trx }}</div>
                <div class="trend">Transaksi Terdata</div>
            </div>
            <div class="card">
                <div class="card-label">Item Terjual</div>
                <div class="value">{{ total_items }} Pcs</div>
                <div class="trend">Produk Keluar</div>
            </div>
        </section>

        <section class="panel" style="margin-bottom: 25px;">
            <h2>Input Transaksi Penjualan Baru</h2>
            <form method="POST" action="{{ url_for('tambah_penjualan') }}">
                <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
                <div class="form-grid">
                    <div class="form-group">
                        <label>Nama Pelanggan</label>
                        <input type="text" name="nama_pelanggan" required placeholder="Contoh: Ani Rahmawati">
                    </div>
                    <div class="form-group">
                        <label>Nama Produk</label>
                        <input type="text" name="produk" required placeholder="Contoh: Monitor LG 24 Inch">
                    </div>
                    <div class="form-group">
                        <label>Jumlah (Qty)</label>
                        <input type="number" name="jumlah" required value="1" min="1">
                    </div>
                    <div class="form-group">
                        <label>Total Harga (Rp)</label>
                        <input type="number" name="total_harga" required placeholder="Contoh: 1500000">
                    </div>
                    <div class="form-group">
                        <label>Status Transaksi</label>
                        <select name="status">
                            <option value="Selesai">Selesai</option>
                            <option value="Pending">Pending</option>
                            <option value="Batal">Batal</option>
                        </select>
                    </div>
                </div>
                <button type="submit" class="btn-submit">+ Simpan Data Penjualan</button>
            </form>
        </section>

        <section class="panel">
            <h2>Tabel Data Penjualan</h2>
            <table>
                <thead>
                    <tr>
                        <th>KODE TRX</th>
                        <th>TANGGAL</th>
                        <th>NAMA PELANGGAN</th>
                        <th>PRODUK</th>
                        <th>QTY</th>
                        <th>TOTAL HARGA</th>
                        <th>STATUS</th>
                    </tr>
                </thead>
                <tbody>
                    {% for row in sales %}
                    <tr>
                        <td><b>{{ row.kode_trx }}</b></td>
                        <td>{{ row.tanggal }}</td>
                        <td>{{ row.nama_pelanggan }}</td>
                        <td>{{ row.produk }}</td>
                        <td>{{ row.jumlah }}</td>
                        <td>Rp {{ "{:,.0f}".format(row.total_harga) }}</td>
                        <td><b>{{ row.status }}</b></td>
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
        </section>

        <!-- FITUR 3: DATA PELANGGAN -->
        {% elif active_page == 'pelanggan' %}
        <section class="cards">
            <div class="card">
                <div class="card-label">Total Pelanggan Terdaftar</div>
                <div class="value">{{ total_customers }}</div>
                <div class="trend">Pelanggan Aktif</div>
            </div>
        </section>

        <section class="panel" style="margin-bottom: 25px;">
            <h2>Tambah Data Pelanggan Baru</h2>
            <form method="POST" action="{{ url_for('tambah_pelanggan') }}">
                <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
                <div class="form-grid">
                    <div class="form-group">
                        <label>Nama Lengkap</label>
                        <input type="text" name="nama" required placeholder="Contoh: Bambang Pamungkas">
                    </div>
                    <div class="form-group">
                        <label>Alamat Email</label>
                        <input type="email" name="email" required placeholder="Contoh: bambang@gmail.com">
                    </div>
                    <div class="form-group">
                        <label>No. Telepon / WA</label>
                        <input type="text" name="telepon" required placeholder="Contoh: 08123456789">
                    </div>
                    <div class="form-group">
                        <label>Alamat</label>
                        <input type="text" name="alamat" required placeholder="Contoh: Tangerang, Banten">
                    </div>
                </div>
                <button type="submit" class="btn-submit">+ Tambah Pelanggan</button>
            </form>
        </section>

        <section class="panel">
            <h2>Direktori Data Pelanggan</h2>
            <table>
                <thead>
                    <tr>
                        <th>KODE</th>
                        <th>NAMA PELANGGAN</th>
                        <th>EMAIL</th>
                        <th>TELEPON</th>
                        <th>ALAMAT</th>
                        <th>TANGGAL DAFTAR</th>
                    </tr>
                </thead>
                <tbody>
                    {% for c in customers %}
                    <tr>
                        <td><b>{{ c.kode_pelanggan }}</b></td>
                        <td>{{ c.nama }}</td>
                        <td>{{ c.email }}</td>
                        <td>{{ c.telepon }}</td>
                        <td>{{ c.alamat }}</td>
                        <td>{{ c.tanggal_daftar }}</td>
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
        </section>

        <!-- PERUBAHAN FITUR 4: KELOLA ADMIN USERS -->
        {% elif active_page == 'admin_users' %}
        <section class="cards">
            <div class="card">
                <div class="card-label">Total Administrator</div>
                <div class="value">{{ total_admins }}</div>
                <div class="trend">Pengguna Aktif Memiliki Akses</div>
            </div>
        </section>

        <section class="panel" style="margin-bottom: 25px;">
            <h2>Tambah Akun Administrator Baru</h2>
            <form method="POST" action="{{ url_for('tambah_admin') }}">
                <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
                <div class="form-grid">
                    <div class="form-group">
                        <label>Username Admin</label>
                        <input type="text" name="username" required placeholder="Contoh: admin_sales">
                    </div>
                    <div class="form-group">
                        <label>Password</label>
                        <input type="password" name="password" required placeholder="Masukkan password">
                    </div>
                </div>
                <button type="submit" class="btn-submit">+ Buat Akun Admin</button>
            </form>
        </section>

        <section class="panel">
            <h2>Daftar Pengguna Administrator</h2>
            <table>
                <thead>
                    <tr>
                        <th>ID</th>
                        <th>USERNAME</th>
                        <th>STATUS HAK AKSES</th>
                    </tr>
                </thead>
                <tbody>
                    {% for a in admins %}
                    <tr>
                        <td><b>{{ a.id }}</b></td>
                        <td>{{ a.username }}</td>
                        <td><span style="color:#22543d; font-weight:bold;">Aktif (Full Access)</span></td>
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
        </section>

        <!-- FITUR 5: SISTEM PENGATURAN -->
        {% elif active_page == 'settings' %}
        <section class="panel" style="margin-bottom:25px; margin-top:25px;">
            <h2>Pengaturan Aplikasi</h2>
            <form method="POST" action="{{ url_for('simpan_settings') }}">
                <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
                <div class="form-grid">
                    <div class="form-group">
                        <label>Nama Portal / Aplikasi</label>
                        <input type="text" name="nama_aplikasi" value="{{ app_settings.nama_aplikasi }}" required>
                    </div>
                    <div class="form-group">
                        <label>Versi Sistem</label>
                        <input type="text" name="versi" value="{{ app_settings.versi }}" required>
                    </div>
                    <div class="form-group" style="grid-column: span 2;">
                        <label>Footer Teks</label>
                        <input type="text" name="footer_text" value="{{ app_settings.footer_text }}" required>
                    </div>
                </div>
                <button type="submit" class="btn-submit">Simpan Pengaturan Aplikasi</button>
            </form>
        </section>

        <section class="panel">
            <h2>Keamanan - Ubah Password Admin Saya</h2>
            <form method="POST" action="{{ url_for('ubah_password') }}">
                <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
                <div class="form-grid">
                    <div class="form-group">
                        <label>Password Lama</label>
                        <input type="password" name="password_lama" required placeholder="Masukkan password lama">
                    </div>
                    <div class="form-group">
                        <label>Password Baru</label>
                        <input type="password" name="password_baru" required placeholder="Masukkan password baru">
                    </div>
                </div>
                <button type="submit" class="btn-submit" style="background:#e53e3e;">Ubah Password</button>
            </form>
        </section>
        {% endif %}
    </main>
</div>
</body>
</html>
"""


# =========================================================
# 6. ROUTE FLASK (OVERVIEW, PENJUALAN, PELANGGAN, ADMINS)
# =========================================================

def get_settings():
    with get_db() as conn:
        return conn.execute("SELECT * FROM settings WHERE id = 1").fetchone()


@app.route("/", methods=["GET", "POST"])
def login():
    if current_user.is_authenticated:
        return redirect(url_for("dashboard"))

    if request.method == "POST":
        username = request.form.get("username", "").strip()
        password = request.form.get("password", "")

        admin = Admin.find_by_username(username)

        if admin and check_password_hash(admin.password_hash, password):
            login_user(admin)
            return redirect(url_for("dashboard"))

        flash("Username atau password salah.")

    return render_template_string(LOGIN_HTML)


# --- ROUTE OVERVIEW UTAMA ---
@app.route("/dashboard")
@login_required
def dashboard():
    with get_db() as conn:
        recent_sales = conn.execute("SELECT * FROM penjualan ORDER BY id DESC LIMIT 5").fetchall()
        total_omset = conn.execute("SELECT SUM(total_harga) FROM penjualan").fetchone()[0] or 0
        total_trx = conn.execute("SELECT COUNT(id) FROM penjualan").fetchone()[0] or 0
        total_customers = conn.execute("SELECT COUNT(id) FROM pelanggan").fetchone()[0] or 0

    return render_template_string(
        BASE_LAYOUT,
        active_page="overview",
        kicker="OVERVIEW",
        title="Dashboard Utama",
        description="Ringkasan aktivitas performa sistem penjualan dan pelanggan.",
        recent_sales=recent_sales,
        total_omset=total_omset,
        total_trx=total_trx,
        total_customers=total_customers,
        app_settings=get_settings()
    )


# --- ROUTE DATA PENJUALAN ---
@app.route("/penjualan")
@login_required
def penjualan():
    with get_db() as conn:
        sales = conn.execute("SELECT * FROM penjualan ORDER BY id DESC").fetchall()
        total_omset = conn.execute("SELECT SUM(total_harga) FROM penjualan").fetchone()[0] or 0
        total_trx = conn.execute("SELECT COUNT(id) FROM penjualan").fetchone()[0] or 0
        total_items = conn.execute("SELECT SUM(jumlah) FROM penjualan").fetchone()[0] or 0

    return render_template_string(
        BASE_LAYOUT,
        active_page="penjualan",
        kicker="DATA PENJUALAN",
        title="Web Data Penjualan",
        description="Kelola dan catat transaksi penjualan produk secara realtime.",
        sales=sales,
        total_omset=total_omset,
        total_trx=total_trx,
        total_items=total_items,
        app_settings=get_settings()
    )


@app.route("/tambah-penjualan", methods=["POST"])
@login_required
def tambah_penjualan():
    nama_pelanggan = request.form.get("nama_pelanggan")
    produk = request.form.get("produk")
    jumlah = int(request.form.get("jumlah", 1))
    total_harga = float(request.form.get("total_harga", 0))
    status = request.form.get("status", "Selesai")
    tanggal = time.strftime("%Y-%m-%d")

    with get_db() as conn:
        count = conn.execute("SELECT COUNT(id) FROM penjualan").fetchone()[0] + 1
        kode_trx = f"#TRX-{count:03d}"
        conn.execute("""
            INSERT INTO penjualan (kode_trx, nama_pelanggan, produk, jumlah, total_harga, tanggal, status)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (kode_trx, nama_pelanggan, produk, jumlah, total_harga, tanggal, status))

    flash("Data Penjualan Berhasil Ditambahkan!")
    return redirect(url_for("penjualan"))


# --- ROUTE DATA PELANGGAN ---
@app.route("/pelanggan")
@login_required
def pelanggan():
    with get_db() as conn:
        customers = conn.execute("SELECT * FROM pelanggan ORDER BY id DESC").fetchall()
        total_customers = conn.execute("SELECT COUNT(id) FROM pelanggan").fetchone()[0] or 0

    return render_template_string(
        BASE_LAYOUT,
        active_page="pelanggan",
        kicker="DATA PELANGGAN",
        title="Manajemen Data Pelanggan",
        description="Kelola informasi seluruh pelanggan terdaftar.",
        customers=customers,
        total_customers=total_customers,
        app_settings=get_settings()
    )


@app.route("/tambah-pelanggan", methods=["POST"])
@login_required
def tambah_pelanggan():
    nama = request.form.get("nama")
    email = request.form.get("email")
    telepon = request.form.get("telepon")
    alamat = request.form.get("alamat")
    tanggal_daftar = time.strftime("%Y-%m-%d")

    with get_db() as conn:
        count = conn.execute("SELECT COUNT(id) FROM pelanggan").fetchone()[0] + 1
        kode_pelanggan = f"CUST-{count:03d}"
        conn.execute("""
            INSERT INTO pelanggan (kode_pelanggan, nama, email, telepon, alamat, tanggal_daftar)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (kode_pelanggan, nama, email, telepon, alamat, tanggal_daftar))

    flash("Data Pelanggan Berhasil Ditambahkan!")
    return redirect(url_for("pelanggan"))


# --- ROUTE KELOLA ADMIN USERS ---
@app.route("/admin-users")
@login_required
def admin_users():
    with get_db() as conn:
        admins = conn.execute("SELECT id, username FROM admins ORDER BY id ASC").fetchall()
        total_admins = len(admins)

    return render_template_string(
        BASE_LAYOUT,
        active_page="admin_users",
        kicker="ADMIN USERS",
        title="Kelola Pengguna Admin",
        description="Manajemen akun administrator dengan hak akses kontrol portal.",
        admins=admins,
        total_admins=total_admins,
        app_settings=get_settings()
    )


@app.route("/tambah-admin", methods=["POST"])
@login_required
def tambah_admin():
    username = request.form.get("username", "").strip()
    password = request.form.get("password", "")

    if username and password:
        password_hash = generate_password_hash(password)
        try:
            with get_db() as conn:
                conn.execute("INSERT INTO admins (username, password_hash) VALUES (?, ?)", (username, password_hash))
            flash(f"Akun admin '{username}' berhasil ditambahkan!")
        except sqlite3.IntegrityError:
            flash("Username sudah digunakan!")
    else:
        flash("Username dan password tidak boleh kosong!")

    return redirect(url_for("admin_users"))


# --- ROUTE SISTEM PENGATURAN ---
@app.route("/settings")
@login_required
def settings():
    return render_template_string(
        BASE_LAYOUT,
        active_page="settings",
        kicker="SISTEM PENGATURAN",
        title="Pengaturan Sistem",
        description="Pengaturan preferensi aplikasi dan akun admin.",
        app_settings=get_settings()
    )


@app.route("/simpan-settings", methods=["POST"])
@login_required
def simpan_settings():
    nama_aplikasi = request.form.get("nama_aplikasi")
    versi = request.form.get("versi")
    footer_text = request.form.get("footer_text")

    with get_db() as conn:
        conn.execute("""
            UPDATE settings
            SET nama_aplikasi = ?, versi = ?, footer_text = ?
            WHERE id = 1
        """, (nama_aplikasi, versi, footer_text))

    flash("Pengaturan Sistem Berhasil Diperbarui!")
    return redirect(url_for("settings"))


@app.route("/ubah-password", methods=["POST"])
@login_required
def ubah_password():
    password_lama = request.form.get("password_lama")
    password_baru = request.form.get("password_baru")

    with get_db() as conn:
        admin_row = conn.execute("SELECT * FROM admins WHERE id = ?", (current_user.id,)).fetchone()
        if admin_row and check_password_hash(admin_row["password_hash"], password_lama):
            new_hash = generate_password_hash(password_baru)
            conn.execute("UPDATE admins SET password_hash = ? WHERE id = ?", (new_hash, current_user.id))
            flash("Password Admin Berhasil Diubah!")
        else:
            flash("Password lama salah!")

    return redirect(url_for("settings"))


@app.route("/logout")
@login_required
def logout():
    logout_user()
    return redirect(url_for("login"))


# =========================================================
# 7. MENJALANKAN SERVER DAN TAMPILAN KLIK COLAB
# =========================================================

def start_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

if not globals().get("URWEB_server_started", False):
    server_thread = threading.Thread(target=start_flask, daemon=True)
    server_thread.start()
    globals()["URWEB_server_started"] = True
    time.sleep(2)

proxy_url = output.eval_js("google.colab.kernel.proxyPort(5000)")

display(HTML(f"""
<div style="font-family:Arial,sans-serif;padding:20px;
background:#111d31;border:1px solid #304361;border-radius:14px;
color:#f4f7ff;max-width:600px;margin-top:10px;">
  <h3 style="margin-top:0;color:#5ddce4">Web Admin Portal Active</h3>
  <p style="margin-bottom:15px;color:#cbd5e1;">Server berhasil berjalan di Colab. Klik tombol berikut untuk membuka aplikasi:</p>
  <a href="{proxy_url}" target="_blank"
     style="display:inline-block;padding:12px 24px;background:#3978f6;
     color:white;text-decoration:none;border-radius:10px;font-weight:bold;
     box-shadow: 0 4px 12px rgba(57, 120, 246, 0.4);">
     Buka Web Dashboard Admin Portal &#8599;
  </a>
</div>
"""))
import pandas as pd
from datetime import datetime


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
